# SI4006 · Sesión 4 — Lab: **Fine-tuning con LoRA**

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 1 — Transformers

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

Esta semana es **asincrónica** (festivo). Este notebook es un ejemplo **completo y funcional** de
fine-tuning con **LoRA**: toma un modelo pequeño, lo afina sobre un dataset real y lo **compara**
contra un baseline. Córranlo entero, entiéndanlo, y después **cámbienlo por el modelo y el dataset de
su proyecto** para su entrega **M1**.

> **Activen la GPU gratis:** `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`. Con LoRA y un
> modelo pequeño esto entrena en pocos minutos.

## 0 · Setup

Colab 2026 ya trae `transformers` (5.x) y `torch`. **No los fijamos** — solo instalamos lo que falta:
`peft` (LoRA), `datasets`, `evaluate`, `accelerate`.

> **Un detalle de Colab:** trae un `torchao` viejo (0.10) que hace chocar a `peft` al llamar
> `get_peft_model`. No usamos `torchao` en este notebook (es para cuantización / QLoRA, más adelante),
> así que lo quitamos y listo. Si prefieren, la alternativa es `%pip install -q -U torchao`.

In [ ]:
# Instalamos SOLO lo que falta. No fijamos transformers/torch (usamos los de Colab).
%pip install -q peft datasets evaluate accelerate
# Quitamos el torchao viejo de Colab (choca con peft en get_peft_model; no lo usamos aquí).
%pip uninstall -y torchao
print('\nListo.')

In [ ]:
import torch, transformers, peft
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '| peft', peft.__version__)
print('torch', torch.__version__, '| device:', device)
if device == 'cpu':
    print('\n⚠️  Estás en CPU. Funciona, pero entrena lento. Activa la GPU T4 (ver arriba).')

## 1 · Los datos: cargar y hacer el split

Usamos **IMDb** (reseñas de cine en inglés, etiqueta `positiva`/`negativa`) como ejemplo, recortado
para que corra rápido en Colab gratis. **Este es el pedazo que ustedes cambian por su dataset.**

> **Regla de oro:** `train` para entrenar, `validation` para medir. Nunca se mezclan.
> En su M1 necesitan además un `test` real. Aquí, por simplicidad del ejemplo, usamos train + validation.

In [ ]:
from datasets import load_dataset

# Recortamos: 2000 para entrenar, 500 para validar. Suben estos números si tienen GPU y tiempo.
# Ojo: el datasets nuevo exige el id CON namespace ('stanfordnlp/imdb'), ya no el alias corto 'imdb'.
raw = load_dataset('stanfordnlp/imdb')
train_ds = raw['train'].shuffle(seed=42).select(range(2000))
val_ds   = raw['test'].shuffle(seed=42).select(range(500))

print(train_ds)
print('\nUn ejemplo real:')
print('  texto  :', train_ds[0]['text'][:200], '...')
print('  label  :', train_ds[0]['label'], '(0=negativa, 1=positiva)')

## 2 · Modelo base + tokenizer, y tokenización

Cargamos **DistilBERT** (encoder, ~67M, pequeño y rápido) con una cabeza de clasificación de 2 clases,
y su tokenizer. Después convertimos el texto en tokens (números) que el modelo entiende.

> **Para su proyecto:** si su tarea es *clasificar*, cambien el modelo por el que eligieron en S03
> (p. ej. `distilbert-base-multilingual-cased` para español) y ajusten `num_labels`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODELO = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODELO)

def tokenizar(batch):
    return tokenizer(batch['text'], truncation=True, max_length=256)

train_tok = train_ds.map(tokenizar, batched=True)
val_tok   = val_ds.map(tokenizar, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODELO, num_labels=2)
model.to(device)
print('Modelo y tokenizer listos.')

## 3 · El baseline (contra qué comparamos)

Un número solo no dice nada. Antes de entrenar, medimos un **baseline honesto**: la **clase mayoritaria**
(el clasificador tonto que siempre responde la etiqueta más común). Si nuestro modelo afinado no le gana
a esto, no aprendió nada.

> En M1 el baseline recomendado es **el mismo modelo base SIN fine-tuning** (zero-shot). La clase
> mayoritaria es el mínimo; muéstrenla siempre para saber el piso.

In [ ]:
from collections import Counter

conteo = Counter(val_ds['label'])
clase_mayoritaria = conteo.most_common(1)[0][0]
acc_baseline = conteo[clase_mayoritaria] / len(val_ds)
print(f'Distribución en validación: {dict(conteo)}')
print(f'Baseline (clase mayoritaria = {clase_mayoritaria}): accuracy = {acc_baseline:.3f}')

## 4 · LoRA: fine-tuning eficiente

Aquí está el corazón del módulo. En vez de mover **todos** los pesos de DistilBERT, LoRA **congela** el
modelo y aprende dos matrices pequeñas. Miren al final cuántos parámetros se entrenan de verdad: **<1%**.

- `r` (rank): tamaño de las matrices pequeñas.
- `lora_alpha`: cuánto pesa el ajuste (regla común: `alpha ≈ 2·r`).
- `target_modules`: a qué capas se aplica. En DistilBERT, las de atención son `q_lin` y `v_lin`.

> **Ojo:** `target_modules` cambia según el modelo. Para BERT/RoBERTa suele ser `['query','value']`;
> para modelos tipo LLaMA/Qwen, `['q_proj','v_proj']`. Si se equivocan, `peft` avisa.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q_lin', 'v_lin'],   # capas de atención de DistilBERT
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()   # miren el % de parámetros que se entrena

## 5 · Entrenar con la Trainer API

La `Trainer` nos regala el loop de entrenamiento. Le damos el modelo, los datos, la métrica y unos
hiperparámetros mínimos. Entrena 2 épocas — con LoRA eso basta para ver el salto sobre el baseline.

In [ ]:
import numpy as np, evaluate
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

accuracy = evaluate.load('accuracy')
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

args = TrainingArguments(
    output_dir='./s04_lora_out',
    learning_rate=2e-4,               # LoRA aguanta un LR más alto que el full fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy='epoch',            # en transformers 5.x el arg es eval_strategy
    logging_steps=25,
    seed=42,
    report_to='none',                 # cámbienlo a 'wandb' si quieren tracking (ver guía, sección 5)
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,       # en transformers 5.x reemplaza a tokenizer=
    data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer.train()

## 6 · Evaluar y comparar contra el baseline

El momento de la verdad: ¿le ganamos al baseline? Esto es exactamente la **tabla de resultados** que
les piden en M1 — el *delta* que aportó el fine-tuning.

In [ ]:
metrics = trainer.evaluate()
acc_finetune = metrics['eval_accuracy']

print('=== RESULTADOS ===')
print(f'Baseline (clase mayoritaria) : {acc_baseline:.3f}')
print(f'DistilBERT + LoRA (afinado)  : {acc_finetune:.3f}')
print(f'Delta (lo que aportó afinar) : {acc_finetune - acc_baseline:+.3f}')

## 7 · Ejemplos cualitativos (qué hace el modelo)

Los números no bastan: miren **qué responde** el modelo en casos concretos. M1 pide al menos 3.

In [ ]:
import torch
model.eval()
ejemplos = [
    'An absolute masterpiece, I loved every minute of it.',
    'Terrible acting and a boring, predictable plot.',
    'It was okay, not great but not the worst either.',
]
nombres = {0: 'negativa', 1: 'positiva'}
for texto in ejemplos:
    inp = tokenizer(texto, return_tensors='pt', truncation=True, max_length=256).to(device)
    with torch.no_grad():
        pred = model(**inp).logits.argmax(-1).item()
    print(f'[{nombres[pred]:<9}] {texto}')

## 8 · Guardar el adaptador LoRA (opcional)

LoRA solo guarda las matrices pequeñas: son unos pocos MB, no el modelo entero.

In [ ]:
model.save_pretrained('./s04_lora_adapter')
print('Adaptador LoRA guardado en ./s04_lora_adapter (solo los pesos de LoRA).')

---

# Cómo adaptar esto a **las otras dos familias**

El ejemplo de arriba es la ruta **encoder (clasificación)**. Si su proyecto **genera** o **transforma**
texto, la estructura es la misma (datos → LoRA → Trainer → evaluar contra baseline), pero cambian
la clase del modelo, el Trainer y la métrica. Estos son los mapas — adáptenlos, no se corren tal cual.

### A) Familia **Decoder** (Qwen, TinyLlama) — *generar / conversar*
Se llama **SFT** (supervised fine-tuning): pares `instrucción → respuesta`.
```python
from transformers import AutoModelForCausalLM
# modelo = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
# LoraConfig: task_type=TaskType.CAUSAL_LM, target_modules=['q_proj','v_proj']
# Lo más cómodo: SFTTrainer de la librería `trl` (pip install trl), que arma el formato de chat.
# Métrica: cualitativa + perplejidad; ROUGE si hay respuesta de referencia (se profundiza en M2).
```

### B) Familia **Encoder-decoder** (T5, BART) — *transformar (resumir, reescribir, traducir)*
Pares `entrada → salida`.
```python
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
# modelo = AutoModelForSeq2SeqLM.from_pretrained('t5-small')
# LoraConfig: task_type=TaskType.SEQ_2_SEQ_LM, target_modules=['q','v']
# Tokenizar entrada Y salida (usar text_target=... en el tokenizer).
# Trainer: Seq2SeqTrainer con predict_with_generate=True. Métrica: ROUGE (evaluate.load('rouge')).
```

> **Baseline en generación/transformación:** el mismo modelo base **sin afinar** (zero-shot). Comparen
> su salida contra la del modelo afinado sobre el mismo conjunto. La idea es siempre mostrar el *delta*.

---

## Su siguiente paso (M1)
1. Consigan/armen su dataset — vean **`SI4006_Guia_Datos_y_Datos_Sinteticos.md`**.
2. Copien este notebook y cambien **modelo + dataset + métrica** por los de su proyecto.
3. Midan contra un baseline y escriban la lectura honesta.
4. Sigan la asignación **`STAI_M1_Asignacion_Fine-tuning_baseline.md`** y su rúbrica.

*SI4006 · Universidad EAFIT · Sesión 4 — Fine-tuning con LoRA.*